# Virtual Try-On server (CatVTON on a Colab GPU)

This notebook turns a free Colab GPU into the try-on backend for the Streamlit app on your laptop.

**Steps on Google Colab**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. `Runtime → Run all`. Each new session downloads about 6 GB of model weights (a few minutes).
3. Copy the **URL** and **API key** printed in step 4 into the app's sidebar.
4. Keep this tab open while you use the app. If Colab disconnects, run all cells again (the URL changes).

**Steps on Kaggle** (separate free GPU quota, useful when Colab says "Cannot connect to GPU backend")
1. kaggle.com → `Create → New Notebook` → `File → Import Notebook` → upload this `.ipynb`.
2. Right-hand panel → `Session options`: **Accelerator: GPU T4 x2** (or P100) and **Internet: on**
   (Internet needs a phone-verified Kaggle account).
3. `Run all`, then use the URL and key from step 4 exactly as on Colab.

The server code below is generated from `tryon_server/` in the project repository by
`scripts/build_colab_notebook.py`. Edit the `.py` files there, not here.

CatVTON (Chong et al., ICLR 2025) is used under CC BY-NC-SA 4.0 (non-commercial, academic use).

## 1. Check the GPU

In [ ]:
import os, sys, torch
os.makedirs("/content", exist_ok=True)   # Colab has it; on Kaggle we create it so the same paths work
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| platform:",
      "Kaggle" if os.path.exists("/kaggle") else "Colab/other")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Session options → Accelerator → GPU. "
                     "Then run all cells again.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB, compute capability {props.major}.{props.minor}")

## 2. Get CatVTON (pinned commit) and write the server code

In [ ]:
%cd /content
!rm -rf /content/third_party/CatVTON && mkdir -p /content/third_party
!git clone -q https://github.com/Zheng-Chong/CatVTON.git /content/third_party/CatVTON
!git -C /content/third_party/CatVTON checkout -q 7818397f25613beedb3d861a34769f607cfcf3b1 && git -C /content/third_party/CatVTON log -1 --format="CatVTON at %h (%cd)"
!mkdir -p /content/vto/tryon_server

In [ ]:
%%writefile /content/vto/tryon_server/__init__.py
"""Try-on inference server. Runs on the GPU machine (Colab), not on the laptop."""


In [ ]:
%%writefile /content/vto/tryon_server/catvton_engine.py
"""CatVTON inference engine.

This module is the only place that touches the pretrained CatVTON code. It runs on the
GPU machine (Google Colab), never on the laptop.

What CatVTON does (pretrained, not ours):
    1. AutoMasker: DensePose (body-part map) + SCHP-LIP + SCHP-ATR (human parsing) are fused
       by hand-written rules into a "clothing-agnostic" mask: the region where the old garment
       is and the new one should go, while face, hands, hair and other garments are protected.
    2. The masked person image and the garment image are encoded by the SD VAE into latents
       and concatenated side by side (along the height axis).
    3. A Stable Diffusion 1.5 inpainting UNet (cross-attention removed, self-attention
       fine-tuned by the CatVTON authors) denoises the concatenated latent for N DDIM steps.
       Self-attention lets the person half attend to the garment half, which replaces the
       explicit warping stage of older try-on methods.
    4. The person half of the latent is decoded back to an image.

What this file adds (ours):
    - Reproducible, explicit loading with a pinned attention checkpoint and dtype selection
      (bf16 on Ampere+, fp16 on T4).
    - Downloads only the weights that are actually used.
    - Optional repainting of unmasked pixels from the original photo (identity/background
      preservation), per-stage timings, peak-GPU-memory measurement, step-level progress
      reporting, and typed errors.
"""

from __future__ import annotations

import contextlib
import logging
import sys
import threading
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import torch
from PIL import Image

log = logging.getLogger("tryon.engine")

GARMENT_TYPES = ("upper", "lower", "overall")
ATTN_VERSIONS = {"mix": "mix-48k-1024", "vitonhd": "vitonhd-16k-512", "dresscode": "dresscode-16k-512"}

ProgressCallback = Callable[[str, int, int], None]  # (stage, step, total_steps)


# ---------------------------------------------------------------------------
# Errors (mapped to HTTP status codes by server.py)
# ---------------------------------------------------------------------------
class EngineError(Exception):
    """Base class for errors the engine reports to the client."""

    http_status = 500


class InvalidRequestError(EngineError):
    http_status = 422


class GPUOutOfMemoryError(EngineError):
    http_status = 507


class GenerationError(EngineError):
    http_status = 500


# ---------------------------------------------------------------------------
# Data classes
# ---------------------------------------------------------------------------
@dataclass
class EngineConfig:
    catvton_repo: Path                         # local clone of github.com/Zheng-Chong/CatVTON
    base_model: str = "booksforcharlie/stable-diffusion-inpainting"  # mirror used by CatVTON's own app.py
    attn_repo: str = "zhengchong/CatVTON"
    attn_version: str = "mix"                  # "mix" = trained on VITON-HD + DressCode + DeepFashion at 1024
    precision: str = "auto"                    # "auto" | "fp16" | "bf16" | "fp32"
    width: int = 768
    height: int = 1024
    skip_safety_check: bool = False
    device: str = "cuda"


@dataclass
class TryOnParams:
    garment_type: str = "upper"
    steps: int = 30
    guidance: float = 2.5
    seed: int = 42
    repaint: bool = True       # copy original pixels back outside the (blurred) mask

    def validate(self) -> None:
        if self.garment_type not in GARMENT_TYPES:
            raise InvalidRequestError(f"garment_type must be one of {GARMENT_TYPES}, got '{self.garment_type}'")
        if not 5 <= self.steps <= 100:
            raise InvalidRequestError(f"steps must be between 5 and 100, got {self.steps}")
        if not 0.0 <= self.guidance <= 7.5:
            raise InvalidRequestError(f"guidance must be between 0 and 7.5, got {self.guidance}")
        if not -1 <= self.seed <= 2**31 - 1:
            raise InvalidRequestError("seed must be -1 (random) or a non-negative 32-bit integer")


@dataclass
class TryOnOutput:
    result: Image.Image
    mask: Image.Image
    person_input: Image.Image          # the exact 768x1024 person image the model saw
    garment_input: Image.Image         # the exact padded garment image the model saw
    timings: dict = field(default_factory=dict)
    peak_gpu_memory_gb: Optional[float] = None
    seed_used: int = 0


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
@contextlib.contextmanager
def trusted_torch_load():
    """Temporarily make torch.load default to weights_only=False.

    PyTorch >= 2.6 changed the default of torch.load to weights_only=True. CatVTON's SCHP loader
    calls torch.load(path) without the argument, and the official SCHP checkpoints may contain
    non-tensor Python objects. We only load the official checkpoints downloaded from the
    authors' Hugging Face repository, so disabling the restriction inside this block is safe.
    """
    original = torch.load

    def patched(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return original(*args, **kwargs)

    torch.load = patched
    try:
        yield
    finally:
        torch.load = original


def select_dtype(precision: str, device: str) -> torch.dtype:
    if precision == "fp32" or device == "cpu":
        return torch.float32
    if precision == "fp16":
        return torch.float16
    if precision == "bf16":
        return torch.bfloat16
    # auto: bf16 needs compute capability >= 8.0 (Ampere). A Colab T4 is 7.5 -> fp16.
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16


# ---------------------------------------------------------------------------
# Engine
# ---------------------------------------------------------------------------
class CatVTONEngine:
    name = "CatVTON"

    def __init__(self, config: EngineConfig):
        self.config = config
        self._lock = threading.Lock()   # one GPU, one generation at a time
        self.dtype = select_dtype(config.precision, config.device)

        repo = Path(config.catvton_repo).resolve()
        if not (repo / "model" / "pipeline.py").exists():
            raise FileNotFoundError(f"CatVTON repository not found at {repo}")
        if str(repo) not in sys.path:
            sys.path.insert(0, str(repo))   # CatVTON uses top-level imports: model.*, utils, detectron2, densepose
        # CatVTON's code does `from utils import ...` meaning ITS utils.py. If some other module named
        # `utils` was imported earlier in this process, Python would reuse it and CatVTON would crash
        # with a confusing ImportError. (This is why our own helper package is called `common`.)
        for name in ("utils", "model"):
            loaded = sys.modules.get(name)
            if loaded is not None and not str(getattr(loaded, "__file__", "")).startswith(str(repo)):
                raise ImportError(
                    f"A different module named '{name}' ({getattr(loaded, '__file__', '?')}) is already imported; "
                    "it shadows CatVTON's own module. Rename it or load the engine in a fresh process."
                )

        # Imported here, not at module top, because they only exist after the repo is on sys.path.
        from diffusers.image_processor import VaeImageProcessor
        from huggingface_hub import snapshot_download
        from model.cloth_masker import AutoMasker
        from model.pipeline import CatVTONPipeline
        from utils import repaint_result, resize_and_crop, resize_and_padding

        self._resize_and_crop = resize_and_crop
        self._resize_and_padding = resize_and_padding
        self._repaint_result = repaint_result

        t0 = time.perf_counter()
        sub = ATTN_VERSIONS[config.attn_version]
        log.info("Downloading CatVTON weights (%s, DensePose, SCHP) ...", sub)
        attn_path = snapshot_download(
            repo_id=config.attn_repo,
            allow_patterns=[f"{sub}/*", "DensePose/*", "SCHP/*"],   # skip the two unused attention variants
        )
        log.info("Loading CatVTON pipeline (dtype=%s) ...", self.dtype)
        with trusted_torch_load():
            self.pipeline = CatVTONPipeline(
                base_ckpt=config.base_model,
                attn_ckpt=attn_path,
                attn_ckpt_version=config.attn_version,
                weight_dtype=self.dtype,
                device=config.device,
                skip_safety_check=config.skip_safety_check,
            )
            self.automasker = AutoMasker(
                densepose_ckpt=str(Path(attn_path) / "DensePose"),
                schp_ckpt=str(Path(attn_path) / "SCHP"),
                device=config.device,
            )
        self.mask_processor = VaeImageProcessor(
            vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True
        )
        self.load_seconds = time.perf_counter() - t0
        log.info("CatVTON ready in %.1f s", self.load_seconds)

    # -- public API ---------------------------------------------------------
    def info(self) -> dict:
        info = {
            "engine": self.name,
            "attn_version": self.config.attn_version,
            "base_model": self.config.base_model,
            "dtype": str(self.dtype).replace("torch.", ""),
            "resolution": [self.config.width, self.config.height],
            "garment_types": list(GARMENT_TYPES),
            "safety_checker": not self.config.skip_safety_check,
            "load_seconds": round(self.load_seconds, 1),
        }
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            info["gpu"] = props.name
            info["gpu_memory_gb"] = round(props.total_memory / 1024**3, 1)
        return info

    def generate(
        self,
        person: Image.Image,
        garment: Image.Image,
        params: TryOnParams,
        user_mask: Optional[Image.Image] = None,
        progress: Optional[ProgressCallback] = None,
    ) -> TryOnOutput:
        params.validate()
        progress = progress or (lambda stage, step, total: None)
        with self._lock:
            try:
                return self._generate_locked(person, garment, params, user_mask, progress)
            except torch.cuda.OutOfMemoryError as exc:
                torch.cuda.empty_cache()
                raise GPUOutOfMemoryError(
                    "The GPU ran out of memory. Try fewer steps, or restart the Colab runtime."
                ) from exc
            except EngineError:
                raise
            except Exception as exc:  # anything else from CatVTON / detectron2
                log.exception("Generation failed")
                raise GenerationError(f"Generation failed: {type(exc).__name__}: {exc}") from exc

    # -- internals ----------------------------------------------------------
    def _generate_locked(self, person, garment, params, user_mask, progress) -> TryOnOutput:
        cfg = self.config
        size = (cfg.width, cfg.height)
        timings = {}
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        # 1. Preprocessing: identical to CatVTON's app.py
        t = time.perf_counter()
        person = self._resize_and_crop(person.convert("RGB"), size)       # centre-crop to 3:4, resize
        garment = self._resize_and_padding(garment.convert("RGB"), size)  # fit inside 3:4, pad white
        timings["preprocess_s"] = time.perf_counter() - t

        # 2. Clothing-agnostic mask (DensePose + SCHP), or a user-drawn mask
        progress("masking", 0, params.steps)
        t = time.perf_counter()
        if user_mask is not None:
            mask = self._resize_and_crop(user_mask.convert("L"), size)
            mask = Image.fromarray(np.where(np.array(mask) > 0, 255, 0).astype(np.uint8))
        else:
            mask = self.automasker(person, params.garment_type)["mask"]
        if np.array(mask).max() == 0:
            raise GenerationError(
                "No clothing region was found for the selected garment type. "
                "Check that the person is clearly visible and the garment type is correct."
            )
        blurred_mask = self.mask_processor.blur(mask, blur_factor=9)
        timings["mask_s"] = time.perf_counter() - t

        # 3. Diffusion with step-level progress
        seed = params.seed if params.seed >= 0 else int(torch.randint(0, 2**31 - 1, (1,)).item())
        generator = torch.Generator(device=cfg.device).manual_seed(seed)
        t = time.perf_counter()
        with self._count_steps(params.steps, progress):
            result = self.pipeline(
                image=person,
                condition_image=garment,
                mask=blurred_mask,
                num_inference_steps=params.steps,
                guidance_scale=params.guidance,
                height=cfg.height,
                width=cfg.width,
                generator=generator,
            )[0]
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        timings["diffusion_s"] = time.perf_counter() - t

        # 4. Post-processing
        if params.repaint:
            result = self._repaint_result(result, person, blurred_mask)
        timings["total_s"] = sum(timings.values())
        timings = {k: round(v, 2) for k, v in timings.items()}

        peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None
        progress("done", params.steps, params.steps)
        return TryOnOutput(
            result=result, mask=mask, person_input=person, garment_input=garment,
            timings=timings, peak_gpu_memory_gb=round(peak, 2) if peak else None, seed_used=seed,
        )

    @contextlib.contextmanager
    def _count_steps(self, total: int, progress: ProgressCallback):
        """Report denoising progress by wrapping the scheduler's step() method.

        CatVTONPipeline has no callback argument; every denoising iteration calls
        noise_scheduler.step() exactly once, so counting those calls gives the step number.
        Safe because generate() holds a lock (one job at a time).
        """
        scheduler = self.pipeline.noise_scheduler
        original_step = scheduler.step
        counter = {"n": 0}

        def counting_step(*args, **kwargs):
            out = original_step(*args, **kwargs)
            counter["n"] += 1
            progress("diffusion", counter["n"], total)
            return out

        scheduler.step = counting_step
        try:
            yield
        finally:
            scheduler.step = original_step


In [ ]:
%%writefile /content/vto/tryon_server/jobs.py
"""In-memory job queue for try-on requests.

Why jobs instead of one long HTTP request: the free Cloudflare quick tunnel closes requests
that take longer than ~100 s, and a 50-step 1024x768 generation on a T4 can take about that
long. So the client submits a job, gets an id immediately, and polls for progress/result.
A single worker thread processes jobs in order because there is one GPU.
"""

from __future__ import annotations

import logging
import queue
import threading
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

log = logging.getLogger("tryon.jobs")

QUEUED, RUNNING, DONE, ERROR = "queued", "running", "done", "error"


@dataclass
class Job:
    id: str
    payload: dict
    status: str = QUEUED
    stage: str = "queued"
    step: int = 0
    total_steps: int = 0
    created_at: float = field(default_factory=time.time)
    started_at: Optional[float] = None
    finished_at: Optional[float] = None
    result: Any = None
    error: Optional[str] = None
    error_status: int = 500

    def progress(self, stage: str, step: int, total: int) -> None:
        self.stage, self.step, self.total_steps = stage, step, total

    def public_status(self, queue_position: int | None = None) -> dict:
        status = {
            "job_id": self.id,
            "status": self.status,
            "stage": self.stage,
            "step": self.step,
            "total_steps": self.total_steps,
            "queue_position": queue_position,
        }
        if self.started_at:
            status["elapsed_s"] = round((self.finished_at or time.time()) - self.started_at, 1)
        if self.error:
            status["error"] = self.error
            status["error_status"] = self.error_status
        return status


class JobManager:
    """FIFO queue + one worker thread. Keeps the most recent `max_jobs` jobs in memory."""

    def __init__(self, handler: Callable[[Job], Any], max_queue: int = 8, max_jobs: int = 50):
        self._handler = handler
        self._queue: "queue.Queue[Job]" = queue.Queue(maxsize=max_queue)
        self._jobs: dict[str, Job] = {}
        self._order: list[str] = []
        self._max_jobs = max_jobs
        self._lock = threading.Lock()
        self._worker = threading.Thread(target=self._run, name="tryon-worker", daemon=True)
        self._worker.start()

    def submit(self, payload: dict) -> Job:
        job = Job(id=uuid.uuid4().hex[:12], payload=payload)
        try:
            self._queue.put_nowait(job)
        except queue.Full:
            raise QueueFullError("The server is busy (queue full). Please try again in a minute.")
        with self._lock:
            self._jobs[job.id] = job
            self._order.append(job.id)
            while len(self._order) > self._max_jobs:  # forget old jobs (and their images)
                self._jobs.pop(self._order.pop(0), None)
        return job

    def get(self, job_id: str) -> Optional[Job]:
        with self._lock:
            return self._jobs.get(job_id)

    def queue_position(self, job: Job) -> Optional[int]:
        if job.status != QUEUED:
            return None
        with self._lock:
            waiting = [j for j in (self._jobs.get(i) for i in self._order) if j and j.status == QUEUED]
        return next((pos for pos, j in enumerate(waiting) if j.id == job.id), None)

    def _run(self) -> None:
        while True:
            job = self._queue.get()
            job.status, job.started_at = RUNNING, time.time()
            try:
                job.result = self._handler(job)
                job.status = DONE
            except Exception as exc:
                job.status = ERROR
                job.error = str(exc)
                job.error_status = getattr(exc, "http_status", 500)
                log.warning("Job %s failed: %s", job.id, exc)
            finally:
                job.finished_at = time.time()
                job.payload = {}  # free the uploaded images
                self._queue.task_done()


class QueueFullError(Exception):
    http_status = 503


In [ ]:
%%writefile /content/vto/tryon_server/mock_engine.py
"""TEST DOUBLE - not a try-on method.

A stand-in engine used only by automated tests and for developing the client/UI without a GPU.
It returns a crude 50/50 blend so tests can check the HTTP contract, image round-trips, error
handling and progress reporting. It is never presented as a try-on result: the server reports
engine="MOCK (test only)" in /health and in every response, and the app shows a warning banner.
"""

from __future__ import annotations

import time

from PIL import Image

from .catvton_engine import GARMENT_TYPES, InvalidRequestError, TryOnOutput, TryOnParams


class MockEngine:
    name = "MOCK (test only)"

    def __init__(self, width: int = 768, height: int = 1024, step_delay_s: float = 0.0, fail_with: Exception | None = None):
        self.width, self.height = width, height
        self.step_delay_s = step_delay_s
        self.fail_with = fail_with

    def info(self) -> dict:
        return {"engine": self.name, "resolution": [self.width, self.height], "garment_types": list(GARMENT_TYPES)}

    def generate(self, person, garment, params: TryOnParams, user_mask=None, progress=None) -> TryOnOutput:
        params.validate()
        progress = progress or (lambda *a: None)
        if self.fail_with:
            raise self.fail_with
        size = (self.width, self.height)
        person = person.convert("RGB").resize(size)
        garment = garment.convert("RGB").resize(size)
        for i in range(1, params.steps + 1):
            if self.step_delay_s:
                time.sleep(self.step_delay_s)
            progress("diffusion", i, params.steps)
        mask = Image.new("L", size, 0)
        return TryOnOutput(
            result=Image.blend(person, garment, 0.5), mask=mask,
            person_input=person, garment_input=garment,
            timings={"total_s": 0.0}, seed_used=params.seed,
        )


__all__ = ["MockEngine", "InvalidRequestError"]


In [ ]:
%%writefile /content/vto/tryon_server/server.py
"""HTTP API around the try-on engine (runs on the GPU machine).

Endpoints
    GET  /health          -> engine info, GPU, readiness (no API key needed)
    POST /jobs            -> submit a try-on job (multipart form), returns {job_id}
    GET  /jobs/{job_id}   -> status + progress; includes the result images when done

Images in responses are base64-encoded PNG. Every request except /health must carry the
header `X-API-Key` if the server was started with an API key (the Colab notebook always
generates one, because the tunnel URL is public).

Run (on Colab, from the notebook):
    python -m tryon_server.server --catvton-repo /content/third_party/CatVTON --api-key <key>
"""

from __future__ import annotations

import argparse
import base64
import io
import logging
import secrets
import time
from typing import Optional

from fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image, UnidentifiedImageError

from .catvton_engine import EngineError, TryOnParams
from .jobs import Job, JobManager, QueueFullError

log = logging.getLogger("tryon.server")
API_VERSION = "1.0"


def encode_png(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def decode_upload(data: bytes, field_name: str, max_bytes: int) -> Image.Image:
    if not data:
        raise HTTPException(400, f"The {field_name} image is empty.")
    if len(data) > max_bytes:
        raise HTTPException(413, f"The {field_name} image is larger than {max_bytes // 2**20} MB.")
    try:
        Image.open(io.BytesIO(data)).verify()   # detects truncated/corrupted files
        image = Image.open(io.BytesIO(data))
        image.load()
    except (UnidentifiedImageError, OSError, SyntaxError) as exc:
        raise HTTPException(400, f"The {field_name} file is not a valid image ({exc}).")
    return image


def create_app(engine, api_key: Optional[str] = None, max_upload_mb: int = 15) -> FastAPI:
    app = FastAPI(title="Virtual Try-On Server", version=API_VERSION)
    started = time.time()
    max_bytes = max_upload_mb * 2**20

    def run_job(job: Job):
        p = job.payload
        output = engine.generate(p["person"], p["garment"], p["params"], p.get("mask"), progress=job.progress)
        return {
            "engine": engine.info().get("engine"),
            "result_png": encode_png(output.result),
            "mask_png": encode_png(output.mask),
            "timings": output.timings,
            "peak_gpu_memory_gb": output.peak_gpu_memory_gb,
            "seed_used": output.seed_used,
            "params": vars(p["params"]),
        }

    jobs = JobManager(run_job)

    def require_key(x_api_key: Optional[str] = Header(default=None)):
        if api_key and not (x_api_key and secrets.compare_digest(x_api_key, api_key)):
            raise HTTPException(401, "Missing or wrong API key.")

    @app.exception_handler(QueueFullError)
    async def _queue_full(_, exc: QueueFullError):
        return JSONResponse(status_code=503, content={"detail": str(exc)})

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "api_version": API_VERSION,
            "uptime_s": round(time.time() - started),
            "auth_required": bool(api_key),
            **engine.info(),
        }

    @app.post("/jobs", status_code=202, dependencies=[Depends(require_key)])
    async def submit(
        person: UploadFile = File(...),
        garment: UploadFile = File(...),
        garment_type: str = Form("upper"),
        steps: int = Form(30),
        guidance: float = Form(2.5),
        seed: int = Form(42),
        repaint: bool = Form(True),
        mask: Optional[UploadFile] = File(None),
    ):
        params = TryOnParams(garment_type=garment_type, steps=steps, guidance=guidance, seed=seed, repaint=repaint)
        try:
            params.validate()
        except EngineError as exc:
            raise HTTPException(exc.http_status, str(exc))
        payload = {
            "person": decode_upload(await person.read(), "person", max_bytes),
            "garment": decode_upload(await garment.read(), "garment", max_bytes),
            "params": params,
        }
        if mask is not None:
            mask_bytes = await mask.read()
            if mask_bytes:
                payload["mask"] = decode_upload(mask_bytes, "mask", max_bytes)
        job = jobs.submit(payload)
        return {"job_id": job.id, "status": job.status}

    @app.get("/jobs/{job_id}", dependencies=[Depends(require_key)])
    def job_status(job_id: str):
        job = jobs.get(job_id)
        if job is None:
            raise HTTPException(404, "Unknown job id (the server may have restarted).")
        body = job.public_status(jobs.queue_position(job))
        if job.status == "done":
            body["result"] = job.result
        return body

    return app


def main() -> None:
    parser = argparse.ArgumentParser(description="Virtual try-on inference server")
    parser.add_argument("--engine", choices=["catvton", "mock"], default="catvton")
    parser.add_argument("--catvton-repo", default="third_party/CatVTON")
    parser.add_argument("--attn-version", default="mix", choices=["mix", "vitonhd", "dresscode"])
    parser.add_argument("--precision", default="auto", choices=["auto", "fp16", "bf16", "fp32"])
    parser.add_argument("--skip-safety-check", action="store_true")
    parser.add_argument("--api-key", default=None)
    parser.add_argument("--host", default="0.0.0.0")
    parser.add_argument("--port", type=int, default=8000)
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    if args.engine == "mock":
        from .mock_engine import MockEngine
        engine = MockEngine(step_delay_s=0.05)
        log.warning("Running the MOCK engine: outputs are NOT try-on results.")
    else:
        from pathlib import Path
        from .catvton_engine import CatVTONEngine, EngineConfig
        engine = CatVTONEngine(EngineConfig(
            catvton_repo=Path(args.catvton_repo), attn_version=args.attn_version,
            precision=args.precision, skip_safety_check=args.skip_safety_check,
        ))

    import uvicorn
    uvicorn.run(create_app(engine, api_key=args.api_key), host=args.host, port=args.port, log_level="info")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/vto/tryon_server/requirements-server.txt
# Extra packages the GPU server needs ON TOP of Google Colab's preinstalled stack.
# Deliberately NOT listed: torch, torchvision, diffusers, transformers, accelerate, numpy,
# opencv, pillow, huggingface_hub -> we use Colab's versions (reinstalling torch on Colab
# breaks CUDA builds and wastes ~10 minutes).
#
# Verified on 2026-11-08: CatVTON @ 7818397 imports and runs its full code path
# (DensePose -> SCHP -> mask fusion -> diffusion loop, random weights) on Python 3.12 with
# torch 2.15.0, diffusers 0.42.1, transformers 5.22.2 given these extras.

fvcore           # detectron2 (DensePose) dependency
iopath           # detectron2 file I/O
yacs             # detectron2 config system
omegaconf        # detectron2 lazy configs
pycocotools      # imported by detectron2 data utilities
cloudpickle
av               # imported by densepose visualisation utilities
diffusers>=0.31  # Colab ships it; floor in case an old runtime image is used
fastapi>=0.110
uvicorn>=0.29
python-multipart>=0.0.9


In [ ]:
!pip install -q -r /content/vto/tryon_server/requirements-server.txt
import diffusers, transformers
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)

## 3. Start the server

Loads CatVTON + DensePose + SCHP onto the GPU. **First run: 3–8 minutes** (downloads).
`SAFETY_CHECK` keeps the Stable Diffusion NSFW filter on; set it to `False` only if it fails to load.

In [ ]:
import os, secrets, subprocess, time, requests

SAFETY_CHECK = True
API_KEY = secrets.token_urlsafe(16)
os.chdir("/content/vto")
log_file = open("/content/server.log", "w")
cmd = ["python", "-m", "tryon_server.server", "--catvton-repo", "/content/third_party/CatVTON",
       "--api-key", API_KEY, "--port", "8000"]
if not SAFETY_CHECK:
    cmd.append("--skip-safety-check")
server = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

start = time.time()
while True:
    time.sleep(5)
    if server.poll() is not None:
        print(open("/content/server.log").read()[-4000:])
        raise SystemExit("The server stopped while loading. The log above shows why.")
    try:
        health = requests.get("http://127.0.0.1:8000/health", timeout=5).json()
        break
    except requests.exceptions.RequestException:
        last = open("/content/server.log").read().strip().splitlines()[-1:] or [""]
        print(f"[{time.time() - start:5.0f}s] loading... {last[0][:110]}")
print("Server ready:", health)

## 4. Open a public HTTPS tunnel (Cloudflare quick tunnel, no account needed)

In [ ]:
import re
!wget -q -nc -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        PUBLIC_URL = found.group(0)
        break
if PUBLIC_URL is None:
    print(open("/content/tunnel.log").read()[-3000:])
    raise SystemExit("Tunnel did not start; run this cell again.")
for _ in range(30):   # DNS for a new quick tunnel can take a few seconds
    try:
        if requests.get(PUBLIC_URL + "/health", timeout=10).ok:
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)
print("=" * 70)
print("  Paste these into the app sidebar:")
print(f"  URL     : {PUBLIC_URL}")
print(f"  API key : {API_KEY}")
print("=" * 70)

## 5. Self-test: first real try-on on CatVTON's own demo images

Runs through the same HTTP API the laptop uses, and reports timing and peak GPU memory.
These numbers go into `docs/test_report.md`.

In [ ]:
import base64, io, glob
from PIL import Image
from IPython.display import display

def run_job(person_path, garment_path, garment_type="upper", steps=30, seed=42):
    files = {"person": open(person_path, "rb"), "garment": open(garment_path, "rb")}
    data = {"garment_type": garment_type, "steps": steps, "seed": seed}
    headers = {"X-API-Key": API_KEY}
    job = requests.post("http://127.0.0.1:8000/jobs", files=files, data=data, headers=headers).json()
    while True:
        time.sleep(1)
        status = requests.get(f"http://127.0.0.1:8000/jobs/{job['job_id']}", headers=headers).json()
        if status["status"] in ("done", "error"):
            return status

demo = "/content/third_party/CatVTON/resource/demo/example"
people = sorted(glob.glob(f"{demo}/person/*/*"))
uppers = sorted(glob.glob(f"{demo}/condition/upper/*"))
overalls = sorted(glob.glob(f"{demo}/condition/overall/*"))
tests = [(people[0], uppers[0], "upper", 30), (people[0], uppers[0], "upper", 50), (people[-1], overalls[0], "overall", 30)]
for person_path, garment_path, garment_type, steps in tests:
    status = run_job(person_path, garment_path, garment_type, steps)
    if status["status"] == "error":
        print("ERROR:", status["error"]); continue
    r = status["result"]
    print(f"{garment_type}, {steps} steps -> timings {r['timings']}, peak GPU memory {r['peak_gpu_memory_gb']} GB")
    tiles = [Image.open(person_path).convert("RGB").resize((384, 512)), Image.open(garment_path).convert("RGB").resize((384, 512)),
             Image.open(io.BytesIO(base64.b64decode(r["mask_png"]))).convert("RGB").resize((384, 512)),
             Image.open(io.BytesIO(base64.b64decode(r["result_png"]))).resize((384, 512))]
    sheet = Image.new("RGB", (384 * 4, 512), "white")
    for i, tile in enumerate(tiles):
        sheet.paste(tile, (384 * i, 0))
    display(sheet)

## 6. Server log (run any time to see what the server is doing)

In [ ]:
print(open('/content/server.log').read()[-5000:])

## Troubleshooting
- **App says it cannot reach the server**: Colab disconnected or the tunnel URL changed. Run all cells again and paste the new URL and key.
- **GPU out of memory**: `Runtime → Restart session`, then run all again. Use 30 steps.
- **"Cannot connect to GPU backend" / "No GPU"**: the free Colab GPU quota is used up. Do **not** choose "Connect without GPU"
  (CatVTON on CPU takes many minutes per image). Use Kaggle (steps at the top) or try Colab again later.
- **Kaggle: downloads fail**: Internet is off. Turn it on in Session options (needs phone verification).
- **Safety checker fails to load**: set `SAFETY_CHECK = False` in step 3.